# News Dataset - Exploratory Data Analysis

This notebook explores the news classification dataset and provides insights into:
- Data structure and quality
- Label distribution
- Text characteristics
- Common words and patterns per class

In [ ]:
# Import libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline

## 1. Load Dataset

In [ ]:
from data_loading.news_dataset import NewsDataset

# Load data
dataset = NewsDataset(csv_path='../data/news.csv')
df = dataset.load_data()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Display first few rows
df.head()

## 2. Data Quality Check

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Check for duplicates
duplicates = df.duplicated(subset=['content']).sum()
print(f"\nDuplicate articles: {duplicates}")

In [ ]:
# Data types
df.info()

## 3. Label Distribution

In [ ]:
# Count per class
label_counts = df['label'].value_counts()
print("Label distribution:")
print(label_counts)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
label_counts.plot(kind='bar', ax=ax1, color='skyblue', edgecolor='black')
ax1.set_title('Label Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Label')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Pie chart
label_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', startangle=90)
ax2.set_title('Label Proportions', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Text Length Analysis

In [ ]:
# Calculate text lengths
df['title_length'] = df['title'].str.len()
df['content_length'] = df['content'].str.len()
df['word_count'] = df['content'].str.split().str.len()

# Statistics
print("Text Length Statistics:")
print(df[['title_length', 'content_length', 'word_count']].describe())

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Word count distribution
df['word_count'].hist(bins=50, ax=axes[0, 0], color='lightgreen', edgecolor='black')
axes[0, 0].set_title('Word Count Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Word Count')
axes[0, 0].set_ylabel('Frequency')

# Word count by label
df.boxplot(column='word_count', by='label', ax=axes[0, 1])
axes[0, 1].set_title('Word Count by Label', fontweight='bold')
axes[0, 1].set_xlabel('Label')
axes[0, 1].set_ylabel('Word Count')

# Content length distribution
df['content_length'].hist(bins=50, ax=axes[1, 0], color='lightcoral', edgecolor='black')
axes[1, 0].set_title('Content Length Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Characters')
axes[1, 0].set_ylabel('Frequency')

# Average word count per label
avg_words = df.groupby('label')['word_count'].mean().sort_values()
avg_words.plot(kind='barh', ax=axes[1, 1], color='plum', edgecolor='black')
axes[1, 1].set_title('Average Word Count per Label', fontweight='bold')
axes[1, 1].set_xlabel('Average Word Count')

plt.tight_layout()
plt.show()

## 5. Common Words Analysis

In [ ]:
from preprocessing.text_preprocessing import TextPreprocessor

# Preprocess texts
preprocessor = TextPreprocessor(remove_stopwords=True, lowercase=True)

def get_top_words(texts, n=20):
    """Get top N most common words from texts."""
    all_words = []
    for text in texts:
        cleaned = preprocessor.clean_text(text)
        words = cleaned.split()
        all_words.extend(words)
    
    counter = Counter(all_words)
    return counter.most_common(n)

# Get top words for each class
class_words = {}
for label in df['label'].unique():
    texts = df[df['label'] == label]['content'].values
    class_words[label] = get_top_words(texts, n=15)
    
    print(f"\nTop words for '{label}':")
    for word, count in class_words[label][:10]:
        print(f"  {word}: {count}")

In [ ]:
# Visualize top words per class
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (label, top_words) in enumerate(class_words.items()):
    if idx >= 4:
        break
    
    words = [w[0] for w in top_words]
    counts = [w[1] for w in top_words]
    
    axes[idx].barh(range(len(words)), counts, color='steelblue', edgecolor='black')
    axes[idx].set_yticks(range(len(words)))
    axes[idx].set_yticklabels(words)
    axes[idx].set_xlabel('Frequency')
    axes[idx].set_title(f'Top Words: {label}', fontweight='bold', fontsize=12)
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Sample Articles

In [ ]:
# Show sample articles from each class
for label in df['label'].unique():
    print("\n" + "="*80)
    print(f"Sample from '{label.upper()}' class:")
    print("="*80)
    
    sample = df[df['label'] == label].sample(1).iloc[0]
    print(f"\nTitle: {sample['title']}")
    print(f"\nContent: {sample['content'][:300]}...")
    print(f"\nWord count: {len(sample['content'].split())}")

## 7. Conclusions

Key findings from the EDA:

1. **Dataset Balance**: Check if classes are balanced or imbalanced
2. **Text Length**: Different classes may have different text length patterns
3. **Vocabulary**: Each class has distinctive words and phrases
4. **Quality**: Check for missing values, duplicates, and data quality issues

These insights will inform our modeling decisions:
- If imbalanced: Consider class weights or resampling
- Text length variations: Choose appropriate max_length for neural networks
- Distinctive vocabulary: Good signal for classification